In [ ]:
# Runner Configuration

PROJECT_WORKDIR = "/kaggle/temp/project"
SETUP_COMMAND = None
SOURCE_COMMIT = ""


In [ ]:
# Project Setup

import json
import shutil
import subprocess
from pathlib import Path

project_workdir = Path(PROJECT_WORKDIR)
manifest_files = []

for manifest_path in Path("/kaggle/input").rglob("source_manifest.json"):
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        continue

    if manifest.get("commit") == SOURCE_COMMIT:
        manifest_files.append(manifest_path)

if len(manifest_files) != 1:
    raise RuntimeError(
        "Expected exactly one source dataset matching "
        f"commit {SOURCE_COMMIT}, found: {manifest_files}"
    )

source_path = manifest_files[0].parent

if project_workdir.exists():
    if project_workdir.is_symlink() or project_workdir.is_file():
        project_workdir.unlink()
    else:
        shutil.rmtree(project_workdir)

shutil.copytree(source_path, project_workdir)

if SETUP_COMMAND is not None and SETUP_COMMAND.strip():
    subprocess.run(
        ["bash", "-c", SETUP_COMMAND],
        check=True,
        cwd=project_workdir,
    )

print(f"Source loaded from: {source_path}")
print(f"Project workdir: {project_workdir}")
print("No setup command configured." if SETUP_COMMAND is None or not SETUP_COMMAND.strip() else "Setup completed.")


In [ ]:
# Job Definition

JOB_ID = ""
EXECUTION_ID = ""
SUBMITTED_AT = ""
WORKER_NUMBER = 0
COMMAND = ""


In [ ]:
# Job Execution

import json
import subprocess
from pathlib import Path

job_metadata = {
    "job_id": JOB_ID,
    "execution_id": EXECUTION_ID,
    "submitted_at": SUBMITTED_AT,
    "worker": WORKER_NUMBER,
    "source_commit": SOURCE_COMMIT,
    "command": COMMAND,
}

Path("/kaggle/working/job_metadata.json").write_text(
    json.dumps(job_metadata, indent=2),
    encoding="utf-8",
)

subprocess.run(
    ["bash", "-c", COMMAND],
    check=True,
    cwd=PROJECT_WORKDIR,
)
